# Ticket Text Analysis - Submitter Insights

This notebook analyzes ticket data to understand:
- What types of issues/topics each submitter works on
- Repetitive tasks and common patterns
- Insights for improving utilization and activities

Analysis uses PyTorch and Transformers for deep text understanding.

## 1. Installation and Setup

In [ ]:
# Install required packages
!pip install torch transformers pandas numpy scikit-learn matplotlib seaborn plotly gensim spacy -q

In [ ]:
# Download spacy model for NER
!python -m spacy download en_core_web_sm

## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel, pipeline
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation, PCA
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.cluster import DBSCAN, AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter, defaultdict
import re

## 3. Load and Explore Data

In [ ]:
# Load ticket data
df = pd.read_csv('sample_ticket_data.csv')
print(f"Loaded {len(df)} tickets")
df.head()

In [ ]:
# Check data info
df.info()

In [ ]:
# Check for missing values
df.isnull().sum()

## 4. Data Preprocessing

In [ ]:
# Combine text fields for analysis
df['combined_text'] = (
    df['Company'].fillna('') + ' ' + 
    df['Description'].fillna('') + ' ' + 
    df['Task Description'].fillna('')
)

# Clean text
df['combined_text'] = df['combined_text'].str.strip()
df['combined_text'] = df['combined_text'].str.replace(r'\s+', ' ', regex=True)

In [ ]:
# Check submitters
submitter_counts = df['Submitter'].value_counts()
print(f"Number of unique submitters: {len(submitter_counts)}")
print("\nTickets per submitter:")
submitter_counts

In [ ]:
# Show sample combined text
print("Sample combined text:")
for i in range(min(3, len(df))):
    print(f"\n{i+1}. {df.iloc[i]['combined_text'][:200]}...")

## 5. Transformer-Based Text Embeddings

In [ ]:
# Load pre-trained transformer model (DistilBERT for efficiency on CPU)
model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()
print(f"Loaded {model_name}")

In [ ]:
def get_embeddings(texts, batch_size=8):
    """
    Generate embeddings using transformer model
    Uses mean pooling of last hidden state
    """
    embeddings = []
    
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]
            
            # Tokenize
            encoded = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors='pt'
            )
            
            # Get model output
            outputs = model(**encoded)
            
            # Mean pooling
            attention_mask = encoded['attention_mask']
            token_embeddings = outputs.last_hidden_state
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
            sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
            sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
            batch_embeddings = sum_embeddings / sum_mask
            
            embeddings.append(batch_embeddings.cpu().numpy())
            
            if (i // batch_size + 1) % 5 == 0:
                print(f"Processed {min(i+batch_size, len(texts))}/{len(texts)} texts")
    
    return np.vstack(embeddings)

In [ ]:
# Generate embeddings for all tickets
print("Generating embeddings...")
text_embeddings = get_embeddings(df['combined_text'].tolist())
print(f"Generated embeddings shape: {text_embeddings.shape}")

## 6. Topic Modeling with LDA

In [ ]:
# Prepare text for LDA using TF-IDF
vectorizer = CountVectorizer(
    max_features=1000,
    stop_words='english',
    min_df=2,
    max_df=0.8
)

doc_term_matrix = vectorizer.fit_transform(df['combined_text'])
feature_names = vectorizer.get_feature_names_out()
print(f"Document-term matrix shape: {doc_term_matrix.shape}")

In [ ]:
# Train LDA model
n_topics = 8
lda = LatentDirichletAllocation(
    n_components=n_topics,
    random_state=42,
    max_iter=20,
    learning_method='online'
)

print(f"Training LDA with {n_topics} topics...")
lda_topics = lda.fit_transform(doc_term_matrix)
print("LDA training complete")

In [ ]:
# Display topics
def display_topics(model, feature_names, no_top_words=10):
    topics = {}
    for topic_idx, topic in enumerate(model.components_):
        top_words_idx = topic.argsort()[-no_top_words:][::-1]
        top_words = [feature_names[i] for i in top_words_idx]
        topics[f"Topic {topic_idx}"] = top_words
        print(f"\nTopic {topic_idx}: {', '.join(top_words)}")
    return topics

topic_words = display_topics(lda, feature_names)

In [ ]:
# Assign dominant topic to each ticket
df['dominant_topic'] = lda_topics.argmax(axis=1)
df['topic_probability'] = lda_topics.max(axis=1)

# Topic distribution
topic_dist = df['dominant_topic'].value_counts().sort_index()
print("\nTopic distribution:")
topic_dist

## 7. Entity Extraction with Transformers

In [ ]:
# Load NER pipeline
ner_pipeline = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple",
    device=-1  # CPU
)
print("NER pipeline loaded")

In [ ]:
# Extract technical entities from text
def extract_technical_terms(text):
    """
    Extract technical terms and system names from text
    """
    # Pattern for technical terms
    patterns = [
        r'\b[A-Z]{2,}(?:-[A-Z0-9]+)*\b',  # Acronyms like CPU, RAID, DNS
        r'\b\w+(?:-\w+)*(?:Server|Node|Database|DB)\d*\b',  # System names
        r'\b(?:server|database|network|storage|backup|firewall|router|switch)\b',  # Keywords
    ]
    
    entities = []
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        entities.extend(matches)
    
    return list(set([e.lower() for e in entities]))

# Extract entities for each ticket
df['technical_entities'] = df['combined_text'].apply(extract_technical_terms)

# Show sample
print("Sample technical entities:")
for i in range(min(5, len(df))):
    print(f"\n{i+1}. {df.iloc[i]['technical_entities'][:5]}")

In [ ]:
# Extract entities using transformer NER model (sample for efficiency)
sample_size = min(10, len(df))
sample_texts = df['combined_text'].head(sample_size).tolist()

print(f"\nExtracting named entities from {sample_size} sample tickets...")
ner_results = []
for idx, text in enumerate(sample_texts):
    entities = ner_pipeline(text[:512])  # Limit text length
    ner_results.append(entities)
    if entities:
        print(f"\nTicket {idx+1}:")
        for ent in entities[:3]:  # Show top 3
            print(f"  - {ent['word']}: {ent['entity_group']}")

## 8. Repetitive Task Detection

In [ ]:
# Calculate similarity matrix using embeddings
similarity_matrix = cosine_similarity(text_embeddings)
print(f"Similarity matrix shape: {similarity_matrix.shape}")
print(f"Average similarity: {similarity_matrix.mean():.3f}")

In [ ]:
# Use DBSCAN for clustering similar tasks (not K-means as requested)
from sklearn.preprocessing import StandardScaler

# Reduce dimensionality for clustering
pca = PCA(n_components=50)
embeddings_reduced = pca.fit_transform(text_embeddings)
print(f"Reduced embeddings shape: {embeddings_reduced.shape}")
print(f"Explained variance: {pca.explained_variance_ratio_.sum():.3f}")

In [ ]:
# DBSCAN clustering to find repetitive patterns
dbscan = DBSCAN(eps=2.5, min_samples=2, metric='euclidean')
clusters = dbscan.fit_predict(embeddings_reduced)

df['cluster'] = clusters
print(f"\nFound {len(set(clusters)) - (1 if -1 in clusters else 0)} clusters")
print(f"Outliers (cluster -1): {(clusters == -1).sum()}")

# Show cluster sizes
cluster_counts = pd.Series(clusters).value_counts().sort_index()
print("\nCluster sizes:")
print(cluster_counts.head(10))

In [ ]:
# Analyze repetitive tasks within each cluster
def show_cluster_samples(cluster_id, n=3):
    cluster_tickets = df[df['cluster'] == cluster_id]
    print(f"\n{'='*80}")
    print(f"Cluster {cluster_id} ({len(cluster_tickets)} tickets)")
    print(f"{'='*80}")
    for i, row in cluster_tickets.head(n).iterrows():
        print(f"\n{row['Ticket Number']} - {row['Submitter']}")
        print(f"Description: {row['Description'][:80]}...")
        print(f"Task: {row['Task Description'][:80]}...")

# Show samples from largest clusters
for cluster_id in cluster_counts.head(3).index:
    if cluster_id != -1:  # Skip outliers
        show_cluster_samples(cluster_id)

## 9. Submitter-Level Analysis

In [ ]:
# Analyze each submitter's work patterns
submitter_analysis = {}

for submitter in df['Submitter'].unique():
    submitter_tickets = df[df['Submitter'] == submitter]
    
    analysis = {
        'total_tickets': len(submitter_tickets),
        'topics': submitter_tickets['dominant_topic'].value_counts().to_dict(),
        'companies': submitter_tickets['Company'].value_counts().to_dict(),
        'priorities': submitter_tickets['Priority'].value_counts().to_dict(),
        'effort_types': submitter_tickets['Effort Type'].value_counts().to_dict(),
        'clusters': submitter_tickets['cluster'].value_counts().to_dict(),
    }
    
    submitter_analysis[submitter] = analysis

print(f"Analyzed {len(submitter_analysis)} submitters")

In [ ]:
# Display detailed analysis for top submitters
top_submitters = df['Submitter'].value_counts().head(5).index

for submitter in top_submitters:
    analysis = submitter_analysis[submitter]
    print(f"\n{'='*80}")
    print(f"SUBMITTER: {submitter}")
    print(f"{'='*80}")
    print(f"Total Tickets: {analysis['total_tickets']}")
    print(f"\nTop Topics:")
    for topic, count in sorted(analysis['topics'].items(), key=lambda x: x[1], reverse=True)[:3]:
        print(f"  Topic {topic}: {count} tickets")
    print(f"\nCompanies Served:")
    for company, count in analysis['companies'].items():
        print(f"  {company}: {count} tickets")
    print(f"\nEffort Types:")
    for effort, count in analysis['effort_types'].items():
        print(f"  {effort}: {count} tickets")

In [ ]:
# Identify most common technical entities per submitter
submitter_entities = {}

for submitter in df['Submitter'].unique():
    submitter_tickets = df[df['Submitter'] == submitter]
    all_entities = []
    for entities in submitter_tickets['technical_entities']:
        all_entities.extend(entities)
    
    entity_counts = Counter(all_entities)
    submitter_entities[submitter] = entity_counts.most_common(10)

# Display
for submitter in top_submitters:
    print(f"\n{submitter} - Top Technical Terms:")
    for term, count in submitter_entities[submitter][:5]:
        print(f"  {term}: {count}")

## 10. Repetitive Tasks per Submitter

In [ ]:
# Find repetitive patterns for each submitter
def find_repetitive_tasks(submitter):
    submitter_tickets = df[df['Submitter'] == submitter]
    
    # Get clusters with multiple tickets from this submitter
    cluster_counts = submitter_tickets['cluster'].value_counts()
    repetitive_clusters = cluster_counts[cluster_counts > 1]
    
    print(f"\n{'='*80}")
    print(f"Repetitive Tasks for {submitter}")
    print(f"{'='*80}")
    print(f"Found {len(repetitive_clusters)} repetitive task patterns")
    
    for cluster_id, count in repetitive_clusters.items():
        if cluster_id != -1:  # Skip outliers
            print(f"\nPattern {cluster_id} ({count} occurrences):")
            cluster_tickets = submitter_tickets[submitter_tickets['cluster'] == cluster_id]
            for i, row in cluster_tickets.iterrows():
                print(f"  - {row['Description'][:60]}...")

# Analyze top submitters
for submitter in top_submitters[:3]:
    find_repetitive_tasks(submitter)

## 11. Visualizations

In [ ]:
# Tickets per submitter
plt.figure(figsize=(12, 6))
submitter_counts = df['Submitter'].value_counts()
plt.bar(range(len(submitter_counts)), submitter_counts.values)
plt.xticks(range(len(submitter_counts)), 
           [s.split('@')[0] for s in submitter_counts.index], 
           rotation=45, ha='right')
plt.xlabel('Submitter')
plt.ylabel('Number of Tickets')
plt.title('Tickets per Submitter')
plt.tight_layout()
plt.show()

In [ ]:
# Topic distribution heatmap by submitter
topic_by_submitter = pd.crosstab(df['Submitter'], df['dominant_topic'])

plt.figure(figsize=(12, 8))
sns.heatmap(topic_by_submitter, annot=True, fmt='d', cmap='YlOrRd', cbar_kws={'label': 'Count'})
plt.xlabel('Topic ID')
plt.ylabel('Submitter')
plt.title('Topic Distribution by Submitter')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize embeddings using PCA
pca_viz = PCA(n_components=2)
embeddings_2d = pca_viz.fit_transform(text_embeddings)

plt.figure(figsize=(14, 10))
submitters = df['Submitter'].unique()
colors = plt.cm.tab10(np.linspace(0, 1, len(submitters)))

for idx, submitter in enumerate(submitters):
    mask = df['Submitter'] == submitter
    plt.scatter(
        embeddings_2d[mask, 0],
        embeddings_2d[mask, 1],
        c=[colors[idx]],
        label=submitter.split('@')[0],
        alpha=0.6,
        s=100
    )

plt.xlabel('First Principal Component')
plt.ylabel('Second Principal Component')
plt.title('Ticket Embeddings Visualization by Submitter')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Interactive plot with Plotly
fig = px.scatter(
    x=embeddings_2d[:, 0],
    y=embeddings_2d[:, 1],
    color=df['Submitter'],
    hover_data={'Description': df['Description'].str[:50]},
    labels={'x': 'PC1', 'y': 'PC2'},
    title='Interactive Ticket Embeddings by Submitter'
)
fig.update_traces(marker=dict(size=10, opacity=0.7))
fig.show()

In [ ]:
# Priority distribution by submitter
priority_by_submitter = pd.crosstab(df['Submitter'], df['Priority'])

priority_by_submitter.plot(kind='bar', stacked=True, figsize=(12, 6), colormap='Set2')
plt.xlabel('Submitter')
plt.ylabel('Number of Tickets')
plt.title('Priority Distribution by Submitter')
plt.legend(title='Priority')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 12. Key Insights and Recommendations

In [ ]:
# Generate insights summary
print("="*80)
print("KEY INSIGHTS")
print("="*80)

print(f"\n1. WORKLOAD DISTRIBUTION")
print(f"   - Total submitters: {df['Submitter'].nunique()}")
print(f"   - Total tickets: {len(df)}")
print(f"   - Average tickets per submitter: {len(df) / df['Submitter'].nunique():.1f}")

# Find submitter with most diverse topics
topic_diversity = {}
for submitter in df['Submitter'].unique():
    topics = df[df['Submitter'] == submitter]['dominant_topic'].nunique()
    topic_diversity[submitter] = topics

most_diverse = max(topic_diversity.items(), key=lambda x: x[1])
print(f"\n2. TOPIC DIVERSITY")
print(f"   - Most diverse submitter: {most_diverse[0]}")
print(f"   - Works on {most_diverse[1]} different topics")

# Repetitive work analysis
print(f"\n3. REPETITIVE TASKS")
repetitive_tickets = df[df['cluster'] != -1].groupby('Submitter')['cluster'].apply(
    lambda x: (x.value_counts() > 1).sum()
)
if len(repetitive_tickets) > 0:
    most_repetitive = repetitive_tickets.idxmax()
    print(f"   - Submitter with most repetitive patterns: {most_repetitive}")
    print(f"   - Number of repetitive patterns: {repetitive_tickets[most_repetitive]}")

print(f"\n4. RECOMMENDATIONS")
print(f"   - Automate repetitive tasks to improve efficiency")
print(f"   - Balance workload across submitters")
print(f"   - Create knowledge base for common issue patterns")
print(f"   - Cross-train submitters on diverse topics")

In [ ]:
# Export submitter analysis to CSV
submitter_summary = df.groupby('Submitter').agg({
    'Ticket Number': 'count',
    'Priority': lambda x: x.value_counts().to_dict(),
    'dominant_topic': lambda x: x.value_counts().to_dict(),
    'cluster': lambda x: (x.value_counts() > 1).sum(),  # Count of repetitive patterns
    'Company': lambda x: x.nunique(),
}).rename(columns={
    'Ticket Number': 'total_tickets',
    'cluster': 'repetitive_patterns',
    'Company': 'unique_companies'
})

submitter_summary.to_csv('submitter_analysis_summary.csv')
print("\nSubmitter analysis exported to 'submitter_analysis_summary.csv'")
submitter_summary

## 13. Deep Dive: Specific Submitter Analysis

In [ ]:
# Select a submitter for deep analysis
selected_submitter = df['Submitter'].value_counts().index[0]
print(f"Deep dive analysis for: {selected_submitter}")

submitter_df = df[df['Submitter'] == selected_submitter].copy()
print(f"Total tickets: {len(submitter_df)}")

In [ ]:
# Analyze time patterns
submitter_df['Submit Date'] = pd.to_datetime(submitter_df['Submit Date'], format='%d/%m/%Y %H:%M:%S')
submitter_df['hour'] = submitter_df['Submit Date'].dt.hour
submitter_df['day_of_week'] = submitter_df['Submit Date'].dt.day_name()

print("\nSubmission patterns:")
print(f"Most active hour: {submitter_df['hour'].mode().values[0]}:00")
print(f"Most active day: {submitter_df['day_of_week'].mode().values[0]}")

In [ ]:
# Calculate semantic similarity within submitter's tickets
submitter_indices = df[df['Submitter'] == selected_submitter].index
submitter_embeddings = text_embeddings[submitter_indices]
submitter_similarity = cosine_similarity(submitter_embeddings)

# Find most similar ticket pairs
np.fill_diagonal(submitter_similarity, 0)  # Ignore self-similarity
most_similar_idx = np.unravel_index(submitter_similarity.argmax(), submitter_similarity.shape)
similarity_score = submitter_similarity[most_similar_idx]

print(f"\nMost similar tickets (similarity: {similarity_score:.3f}):")
idx1 = submitter_df.iloc[most_similar_idx[0]]
idx2 = submitter_df.iloc[most_similar_idx[1]]
print(f"\nTicket 1: {idx1['Description']}")
print(f"Ticket 2: {idx2['Description']}")

## 14. Advanced: Topic Evolution Over Time

In [ ]:
# Parse all dates
df['Submit Date'] = pd.to_datetime(df['Submit Date'], format='%d/%m/%Y %H:%M:%S')
df['date'] = df['Submit Date'].dt.date

# Topic trends over time
topic_over_time = df.groupby(['date', 'dominant_topic']).size().reset_index(name='count')

fig = px.line(
    topic_over_time,
    x='date',
    y='count',
    color='dominant_topic',
    title='Topic Trends Over Time',
    labels={'count': 'Number of Tickets', 'date': 'Date', 'dominant_topic': 'Topic'}
)
fig.show()

In [ ]:
# Submitter activity over time
submitter_over_time = df.groupby(['date', 'Submitter']).size().reset_index(name='count')

fig = px.line(
    submitter_over_time,
    x='date',
    y='count',
    color='Submitter',
    title='Submitter Activity Over Time',
    labels={'count': 'Number of Tickets', 'date': 'Date'}
)
fig.show()

## 15. Summary Statistics

In [ ]:
# Create comprehensive summary
summary = {
    'Total Tickets': len(df),
    'Unique Submitters': df['Submitter'].nunique(),
    'Unique Companies': df['Company'].nunique(),
    'Unique Topics Discovered': n_topics,
    'Repetitive Task Clusters': len(set(clusters)) - (1 if -1 in clusters else 0),
    'Avg Similarity Between Tickets': similarity_matrix.mean(),
    'Date Range': f"{df['Submit Date'].min()} to {df['Submit Date'].max()}"
}

print("\n" + "="*80)
print("ANALYSIS SUMMARY")
print("="*80)
for key, value in summary.items():
    print(f"{key:.<40} {value}")

In [ ]:
# Save processed data with all features
output_df = df[[
    'Ticket Number', 'Submitter', 'Company', 'Description', 
    'Task Description', 'Priority', 'dominant_topic', 
    'cluster', 'technical_entities'
]].copy()

output_df.to_csv('processed_tickets_with_analysis.csv', index=False)
print("\nProcessed data saved to 'processed_tickets_with_analysis.csv'")